# CacheBlend All-in-One Colab Notebook

This notebook is self-contained. It includes:
- KV pack/unpack + store
- Adaptive token selector (paper-style thresholds)
- Selective recompute engine
- Blend operation (PyTorch fallback)
- End-to-end CacheBlend pipeline
- Direct smoke tests and a quick TTFT benchmark

Run cells in order.

In [ ]:
# Run once in Colab
!pip install -U pip
!pip install "transformers>=4.45,<5" datasets evaluate rouge_score sentence-transformers bitsandbytes accelerate ninja pytest tqdm

# Optional but recommended for faster HF downloads (set your token in Colab secrets)
# import os
# os.environ["HF_TOKEN"] = "<your_token_here>"

# Verify evaluate loaded (restart kernel and re-run if this errors)
import importlib, evaluate
importlib.reload(evaluate)
print(f'evaluate {evaluate.__version__} ready')


In [ ]:
%%writefile cacheblendplus/kv_store.py
import torch
from typing import Optional

def pack_kv(past_key_values) -> torch.Tensor:
    """HuggingFace past_key_values → (L, 2, N, H, D) float16."""
    if hasattr(past_key_values, 'key_cache') and hasattr(past_key_values, 'value_cache'):
        key_cache = past_key_values.key_cache
        val_cache = past_key_values.value_cache
        layers = []
        for k, v in zip(key_cache, val_cache):
            # k, v are (1, H, N, D)
            k = k.squeeze(0).transpose(0, 1) # (N, H, D)
            v = v.squeeze(0).transpose(0, 1) # (N, H, D)
            layers.append(torch.stack([k, v], dim=0)) # (2, N, H, D)
        return torch.stack(layers, dim=0) # (L, 2, N, H, D)

    # Newer HF cache classes can expose tuple format via a helper.
    if hasattr(past_key_values, "to_legacy_cache"):
        past_key_values = past_key_values.to_legacy_cache()

    layers = []
    for layer_kv in past_key_values:
        if not isinstance(layer_kv, (tuple, list)) or len(layer_kv) < 2:
            raise ValueError("Unsupported cache layer format in past_key_values.")
        # Some HF variants include extra fields per layer. Use first two entries.
        k, v = layer_kv[0], layer_kv[1]
        k = k.squeeze(0).transpose(0, 1)
        v = v.squeeze(0).transpose(0, 1)
        layers.append(torch.stack([k, v], dim=0))
    return torch.stack(layers, dim=0)

def unpack_kv(kv_tensor: torch.Tensor):
    """(L, 2, N, H_kv, D) → DynamicCache for newer transformers."""
    from transformers import DynamicCache
    cache = DynamicCache()
    for i in range(kv_tensor.shape[0]):
        k = kv_tensor[i, 0].permute(1, 0, 2).unsqueeze(0).half()  # (1, H_kv, N, D)
        v = kv_tensor[i, 1].permute(1, 0, 2).unsqueeze(0).half()
        cache.update(k, v, i)
    return cache

def get_model_layers(model) -> list:
    name = type(model).__name__.lower()
    if 'gpt2' in name:
        return list(model.transformer.h)
    if 'mistral' in name or 'llama' in name or 'mistral' in str(type(model.model)):
        return list(model.model.layers)
    raise NotImplementedError(f"Model {type(model).__name__} not supported.")

def get_embeddings(model, input_ids: torch.Tensor) -> torch.Tensor:
    name = type(model).__name__.lower()
    N = input_ids.shape[1]

    if 'gpt2' in name:
        pos_ids = torch.arange(N, device=input_ids.device).unsqueeze(0)
        tok_emb = model.transformer.wte(input_ids)
        pos_emb = model.transformer.wpe(pos_ids)
        return model.transformer.drop(tok_emb + pos_emb)

    if 'mistral' in name or 'llama' in name:
        return model.model.embed_tokens(input_ids)

    raise NotImplementedError

def apply_final_norm(model, hidden: torch.Tensor) -> torch.Tensor:
    name = type(model).__name__.lower()
    if 'gpt2' in name:
        return model.transformer.ln_f(hidden)
    if 'mistral' in name or 'llama' in name:
        return model.model.norm(hidden)
    raise NotImplementedError

class KVCacheStore:
    """
    A simple dictionary-based KV cache store.
    In a real-world scenario, this might use a vector database or disk-based storage.
    """
    def __init__(self):
        self._data = {}

    def store_kv(self, key: str, kv_tensor: torch.Tensor):
        # We store a clone to avoid issues if the original tensor is modified in-place
        self._data[key] = kv_tensor.cpu().clone()

    def load_kv(self, key: str, device: str = "cuda") -> Optional[torch.Tensor]:
        kv = self._data.get(key)
        if kv is not None:
            return kv.to(device)
        return None

    # Aliases to match pipeline.py usage if needed
    def store(self, key: str, kv_tensor: torch.Tensor):
        self.store_kv(key, kv_tensor)
    
    def load(self, key: str, device: str = "cuda") -> Optional[torch.Tensor]:
        return self.load_kv(key, device)



In [ ]:
%%writefile cacheblendplus/token_selector.py
import torch
import torch.nn.functional as F
from typing import List, Tuple, Optional
from .kv_store import pack_kv, unpack_kv, get_model_layers, get_embeddings, apply_final_norm

# ─────────────────────────────────────────────────────────────────────────────
from transformers.models.llama.modeling_llama import apply_rotary_pos_emb, repeat_kv, rotate_half

# ─────────────────────────────────────────────────────────────────────────────
# RoPE Utilities
# ─────────────────────────────────────────────────────────────────────────────

def apply_rope_to_cached_kv(
    k_tensor: torch.Tensor, 
    cos: torch.Tensor, 
    sin: torch.Tensor, 
    indices: Optional[torch.Tensor] = None
) -> torch.Tensor:
    """
    Applies an additional rotation to already-rotated K vectors.
    k_tensor: (N, H, D) or (k, H, D)
    cos, sin: (1, 1, seq_len, D)
    """
    if indices is not None:
        cos = cos[:, :, indices, :]
        sin = sin[:, :, indices, :]
    
    # k_tensor is (N, H, D), we need it to be (1, H, N, D) for apply_rotary_pos_emb
    k_mh = k_tensor.transpose(0, 1).unsqueeze(0)
    
    # apply_rotary_pos_emb expects (batch, heads, seq_len, dim)
    # Since we are applying a DELTA rotation, we use rotate_half manually
    # to avoid the complexities of HuggingFace version differences in apply_rotary_pos_emb
    k_rotated = (k_mh * cos) + (rotate_half(k_mh) * sin)
    
    return k_rotated.squeeze(0).transpose(0, 1)

# ─────────────────────────────────────────────────────────────────────────────
# Layer Execution (In-Place Mutations)
# ─────────────────────────────────────────────────────────────────────────────

def run_layer_full_gpt2_inplace(block, hidden: torch.Tensor, kv_layer_buf: torch.Tensor):
    """
    Run Layer 0 on ALL tokens and write KV directly into the pre-allocated buffer.
    """
    device = hidden.device
    N = hidden.shape[1]
    d = hidden.shape[2]
    n_heads  = block.attn.num_heads
    head_dim = d // n_heads

    ln1_out = block.ln_1(hidden)
    qkv     = block.attn.c_attn(ln1_out)
    Q, K, V = qkv.split(d, dim=-1)

    Q = Q.view(1, N, n_heads, head_dim).transpose(1, 2)
    K = K.view(1, N, n_heads, head_dim).transpose(1, 2)
    V = V.view(1, N, n_heads, head_dim).transpose(1, 2)

    # In-place write to KV buffer to avoid clones later
    kv_layer_buf[0] = K.squeeze(0).transpose(0, 1) # (N, H, D)
    kv_layer_buf[1] = V.squeeze(0).transpose(0, 1)

    mask = torch.tril(torch.ones(N, N, dtype=torch.bool, device=device)).view(1, 1, N, N)

    scale = head_dim ** -0.5
    attn_weights = torch.matmul(Q, K.transpose(-2, -1)) * scale
    attn_weights.masked_fill_(~mask, float('-inf')) # In-place mask
    attn_weights = F.softmax(attn_weights, dim=-1)
    attn_weights = torch.nan_to_num(attn_weights)
    attn_weights = block.attn.attn_dropout(attn_weights)

    attn_out = torch.matmul(attn_weights, V)
    attn_out = attn_out.transpose(1, 2).contiguous().view(1, N, d)
    attn_out = block.attn.c_proj(attn_out)
    attn_out = block.attn.resid_dropout(attn_out)

    hidden_after_attn = hidden + attn_out
    mlp_out           = block.mlp(block.ln_2(hidden_after_attn))

    # Mutate hidden in place
    hidden.copy_(hidden_after_attn + mlp_out)

    # Return fresh K, V as views of the buffer for deviation calculation
    return kv_layer_buf[0], kv_layer_buf[1]


def run_layer_selective_gpt2_inplace(
    block,
    full_hidden    : torch.Tensor,   # (1, N, d) - Mutated in place
    hkvd_indices   : torch.Tensor,   # (k,) int64
    KV_layer_buf   : torch.Tensor,   # (2, N, H, D) - Mutated in place
) -> Tuple[torch.Tensor, torch.Tensor]:
    """
    Run on HKVD tokens. Mutates full_hidden and KV_layer_buf in-place to
    eliminate O(N) memory allocations.
    """
    N = full_hidden.shape[1]
    d = full_hidden.shape[2]
    k = hkvd_indices.shape[0]

    hkvd_hidden = full_hidden[:, hkvd_indices, :]

    ln1_out = block.ln_1(hkvd_hidden)
    qkv     = block.attn.c_attn(ln1_out)
    Q, K_fresh, V_fresh = qkv.split(d, dim=-1)

    n_heads  = block.attn.num_heads
    head_dim = d // n_heads

    Q = Q.view(1, k, n_heads, head_dim).transpose(1, 2)
    K_fresh_mh = K_fresh.view(k, n_heads, head_dim)
    V_fresh_mh = V_fresh.view(k, n_heads, head_dim)

    # Update KV Buffer IN-PLACE with fresh tokens
    KV_layer_buf[0, hkvd_indices] = K_fresh_mh
    KV_layer_buf[1, hkvd_indices] = V_fresh_mh

    # Zero-copy views for attention (now containing the fresh updates)
    K_full = KV_layer_buf[0].transpose(0, 1).unsqueeze(0) # (1, H, N, D)
    V_full = KV_layer_buf[1].transpose(0, 1).unsqueeze(0)

    # Vectorized causal mask generation (avoids Python loops)
    positions = hkvd_indices.unsqueeze(1) # (k, 1)
    idx_matrix = torch.arange(N, device=full_hidden.device).unsqueeze(0) # (1, N)
    mask = (idx_matrix <= positions).view(1, 1, k, N) # (1, 1, k, N)

    scale = head_dim ** -0.5
    attn_weights = torch.matmul(Q, K_full.transpose(-2, -1)) * scale
    attn_weights.masked_fill_(~mask, float('-inf'))
    attn_weights = F.softmax(attn_weights, dim=-1)
    attn_weights = torch.nan_to_num_(attn_weights) # In-place NaN clear
    attn_weights = block.attn.attn_dropout(attn_weights)

    attn_out = torch.matmul(attn_weights, V_full)
    attn_out = attn_out.transpose(1, 2).contiguous().view(1, k, d)
    attn_out = block.attn.c_proj(attn_out)
    attn_out = block.attn.resid_dropout(attn_out)

    hidden_after_attn = hkvd_hidden + attn_out
    mlp_out           = block.mlp(block.ln_2(hidden_after_attn))

    # Scatter updates back into full_hidden IN-PLACE
    full_hidden[0, hkvd_indices, :] = (hidden_after_attn + mlp_out)[0]

    return K_fresh_mh, V_fresh_mh



def run_layer_full_llama_inplace(
    block, hidden: torch.Tensor, kv_layer_buf: torch.Tensor, position_ids: torch.Tensor,
    n_heads: int, n_kv_heads: int, head_dim: int, rotary_emb
):
    """Run Layer 0 on ALL tokens for LLaMA architectures, writing directly to buffer."""
    device = hidden.device
    N = hidden.shape[1]
    d = hidden.shape[2]

    hidden_norm = block.input_layernorm(hidden)

    Q = block.self_attn.q_proj(hidden_norm)
    K = block.self_attn.k_proj(hidden_norm)
    V = block.self_attn.v_proj(hidden_norm)

    Q = Q.view(1, N, n_heads, head_dim).transpose(1, 2)
    K = K.view(1, N, n_kv_heads, head_dim).transpose(1, 2)
    V = V.view(1, N, n_kv_heads, head_dim).transpose(1, 2)

    # Safely extract cos/sin across HuggingFace versions
    try:
        cos, sin = rotary_emb(V, position_ids)
    except TypeError:
        cos, sin = rotary_emb(V, seq_len=N)

    # Safely apply RoPE across HuggingFace versions
    try:
        Q, K = apply_rotary_pos_emb(Q, K, cos, sin, position_ids=position_ids)
    except TypeError:
        Q, K = apply_rotary_pos_emb(Q, K, cos, sin)

    kv_layer_buf[0] = K.squeeze(0).transpose(0, 1)
    kv_layer_buf[1] = V.squeeze(0).transpose(0, 1)

    num_key_value_groups = n_heads // n_kv_heads
    if num_key_value_groups > 1:
        K_full = repeat_kv(K, num_key_value_groups)
        V_full = repeat_kv(V, num_key_value_groups)
    else:
        K_full, V_full = K, V

    mask = torch.tril(torch.ones(N, N, dtype=torch.bool, device=device)).view(1, 1, N, N)

    scale = head_dim ** -0.5
    attn_weights = torch.matmul(Q, K_full.transpose(-2, -1)) * scale
    attn_weights.masked_fill_(~mask, float('-inf'))
    attn_weights = F.softmax(attn_weights, dim=-1, dtype=torch.float32).to(Q.dtype)
    attn_weights = torch.nan_to_num_(attn_weights)

    attn_out = torch.matmul(attn_weights, V_full)
    attn_out = attn_out.transpose(1, 2).contiguous().view(1, N, d)
    attn_out = block.self_attn.o_proj(attn_out)

    hidden_after_attn = hidden + attn_out
    mlp_out = block.mlp(block.post_attention_layernorm(hidden_after_attn))

    hidden.copy_(hidden_after_attn + mlp_out)
    return kv_layer_buf[0], kv_layer_buf[1]


def run_layer_selective_llama_inplace(
    block, full_hidden: torch.Tensor, hkvd_indices: torch.Tensor,
    KV_layer_buf: torch.Tensor, position_ids: torch.Tensor,
    n_heads: int, n_kv_heads: int, head_dim: int, rotary_emb
) -> Tuple[torch.Tensor, torch.Tensor]:
    """Selective execution for LLaMA/Mistral architectures."""
    N = full_hidden.shape[1]
    d = full_hidden.shape[2]
    k = hkvd_indices.shape[0]

    hkvd_hidden = full_hidden[:, hkvd_indices, :]
    hidden_norm = block.input_layernorm(hkvd_hidden)

    Q_fresh = block.self_attn.q_proj(hidden_norm)
    K_fresh = block.self_attn.k_proj(hidden_norm)
    V_fresh = block.self_attn.v_proj(hidden_norm)

    Q_fresh = Q_fresh.view(1, k, n_heads, head_dim).transpose(1, 2)
    K_fresh = K_fresh.view(1, k, n_kv_heads, head_dim).transpose(1, 2)
    V_fresh = V_fresh.view(1, k, n_kv_heads, head_dim).transpose(1, 2)

    hkvd_positions = position_ids[:, hkvd_indices]

    # Safely extract cos/sin across HuggingFace versions
    try:
      cos, sin = rotary_emb(V_fresh, hkvd_positions)
    except TypeError:
      cos, sin = rotary_emb(V_fresh, seq_len=N)
      # Manually index to get cos/sin for HKVD positions only
      if cos.dim() == 3:        # (1, N, D) or (batch, N, D)
          cos = cos[:, hkvd_indices]
          sin = sin[:, hkvd_indices]
      elif cos.dim() == 4:      # (1, 1, N, D)
          cos = cos[:, :, hkvd_indices]
          sin = sin[:, :, hkvd_indices]

    # Safely apply RoPE across HuggingFace versions
    try:
        Q_fresh, K_fresh = apply_rotary_pos_emb(Q_fresh, K_fresh, cos, sin, position_ids=hkvd_positions)
    except TypeError:
        Q_fresh, K_fresh = apply_rotary_pos_emb(Q_fresh, K_fresh, cos, sin)

    K_fresh_mh = K_fresh.squeeze(0).transpose(0, 1)
    V_fresh_mh = V_fresh.squeeze(0).transpose(0, 1)

    KV_layer_buf[0, hkvd_indices] = K_fresh_mh
    KV_layer_buf[1, hkvd_indices] = V_fresh_mh

    K_full = KV_layer_buf[0].transpose(0, 1).unsqueeze(0)
    V_full = KV_layer_buf[1].transpose(0, 1).unsqueeze(0)

    num_key_value_groups = n_heads // n_kv_heads
    if num_key_value_groups > 1:
        K_full = repeat_kv(K_full, num_key_value_groups)
        V_full = repeat_kv(V_full, num_key_value_groups)

    idx_matrix = torch.arange(N, device=full_hidden.device).unsqueeze(0)
    pos_col = hkvd_indices.unsqueeze(1)
    mask = (idx_matrix <= pos_col).view(1, 1, k, N)

    scale = head_dim ** -0.5
    attn_weights = torch.matmul(Q_fresh, K_full.transpose(-2, -1)) * scale
    attn_weights.masked_fill_(~mask, float('-inf'))
    attn_weights = F.softmax(attn_weights, dim=-1, dtype=torch.float32).to(Q_fresh.dtype)
    attn_weights = torch.nan_to_num_(attn_weights)

    attn_out = torch.matmul(attn_weights, V_full)
    attn_out = attn_out.transpose(1, 2).contiguous().view(1, k, d)
    attn_out = block.self_attn.o_proj(attn_out)

    hidden_after_attn = hkvd_hidden + attn_out
    mlp_out = block.mlp(block.post_attention_layernorm(hidden_after_attn))

    full_hidden[0, hkvd_indices, :] = (hidden_after_attn + mlp_out)[0]

    return K_fresh_mh, V_fresh_mh


# ─────────────────────────────────────────────────────────────────────────────
# Systems-Level Deviation & Selection
# ─────────────────────────────────────────────────────────────────────────────

def compute_deviation_l2(
    fresh_K    : torch.Tensor,
    fresh_V    : torch.Tensor,
    cached_K   : torch.Tensor,
    cached_V   : torch.Tensor,
    miss_flags : torch.Tensor,
) -> torch.Tensor:
    """Computes deviation using squared L2 norm efficiently."""
    fK, fV = fresh_K.float(), fresh_V.float()
    cK, cV = cached_K.float(), cached_V.float()
    dev = torch.sum((fK - cK)**2 + (fV - cV)**2, dim=(-2, -1))
    dev = torch.sqrt_(dev)
    dev.masked_fill_(miss_flags, float('inf'))
    return dev

def select_topk(scores: torch.Tensor, k: int) -> torch.Tensor:
    k = min(k, scores.shape[0])
    _, idx = torch.topk(scores, k, sorted=False)
    return torch.sort(idx).values

# ─────────────────────────────────────────────────────────────────────────────
# Fusor Main Engine
# ─────────────────────────────────────────────────────────────────────────────

class CacheBlendFusor:
    def __init__(self, model, r: float = 0.15, r1_factor: float = 1.33):
        self.model    = model
        self.r        = r
        self.r1       = min(r * r1_factor, 1.0)
        self.layers   = get_model_layers(model)
        self.L        = len(self.layers)

        self.n_heads = model.config.num_attention_heads
        self.n_kv_heads = getattr(model.config, "num_key_value_heads", self.n_heads)
        self.head_dim = model.config.hidden_size // self.n_heads

        # Safely extract rotary embeddings for LLaMA architectures
        is_llama = 'llama' in type(self.model).__name__.lower() or 'mistral' in type(self.model).__name__.lower()
        if is_llama:
          if hasattr(self.model.model, 'rotary_emb'):
              self.rotary_emb = self.model.model.rotary_emb
          elif hasattr(self.layers[0].self_attn, 'rotary_emb'):
              self.rotary_emb = self.layers[0].self_attn.rotary_emb
          else:
              raise AttributeError(
                  "Cannot locate rotary_emb — check your transformers version"
              )
        else:
            self.rotary_emb = None

        self.model.eval()

    def fuse(
        self,
        full_input_ids : torch.Tensor,
        KV_new         : torch.Tensor,
        hit_mask       : torch.Tensor,
        chunk_offsets  : Optional[List[int]] = None,
    ) -> Tuple[torch.Tensor, List[torch.Tensor]]:

        N = full_input_ids.shape[1]
        KV_new = KV_new.to(self.model.dtype)
        hkvd_per_layer = []

        miss_count = int((~hit_mask).sum().item())

        k_targets = []
        for i in range(1, self.L):
            progress = i / (self.L - 1) if self.L > 1 else 1.0
            target_ratio = self.r1 + (self.r - self.r1) * progress
            k_targets.append(min(N, miss_count + max(1, int(target_ratio * N))))

        is_llama = self.rotary_emb is not None

        with torch.no_grad():
            # ─────────────────────────────────────────────────────────────────
            # RoPE Correction for non-prefix chunks
            # ─────────────────────────────────────────────────────────────────
            if is_llama and chunk_offsets is not None:
                # We need to rotate the cached KVs if they were pre-computed at a different position.
                # Assuming chunk_offsets[i] is the absolute start position of chunk i in the fused sequence.
                # And assuming chunks were cached starting at position 0.
                for i in range(self.L):
                    for chunk_idx, offset in enumerate(chunk_offsets):
                        if offset == 0: continue # No rotation needed for prefix
                        
                        # Get indices for this chunk
                        chunk_len = (hit_mask.shape[0] // len(chunk_offsets)) # Simplified for demo
                        # In a real impl, we'd have exact start/end per chunk
                        start = offset
                        end = offset + chunk_len
                        
                        # Compute cos/sin for the shift
                        # Since R(a+b) = R(a)R(b), we apply R(offset) to the cached K
                        # which was already R(0)raw_K.
                        pos = torch.arange(start, end, device=KV_new.device).unsqueeze(0)
                        try:
                            cos, sin = self.rotary_emb(KV_new[i, 0, start:end], pos)
                        except TypeError:
                            cos, sin = self.rotary_emb(KV_new[i, 0, start:end], seq_len=end)
                            cos, sin = cos[:, start:end], sin[:, start:end]
                        
                        KV_new[i, 0, start:end] = apply_rope_to_cached_kv(
                            KV_new[i, 0, start:end], cos, sin
                        )

            hidden = get_embeddings(self.model, full_input_ids)

            if is_llama:
                position_ids = torch.arange(N, device=full_input_ids.device).unsqueeze(0)

            # -- LAYER 0 --
            # We keep a copy of the (corrected) cached K, V to compute deviation
            cached_K0 = KV_new[0, 0].clone()
            cached_V0 = KV_new[0, 1].clone()

            if is_llama:
                fresh_K0, fresh_V0 = run_layer_full_llama_inplace(
                    self.layers[0], hidden, KV_new[0], position_ids,
                    self.n_heads, self.n_kv_heads, self.head_dim, self.rotary_emb
                )
            else:
                fresh_K0, fresh_V0 = run_layer_full_gpt2_inplace(self.layers[0], hidden, KV_new[0])

            dev0 = compute_deviation_l2(
                fresh_K0, fresh_V0, cached_K0, cached_V0, ~hit_mask
            )
            k0 = min(N, miss_count + max(1, int(self.r1 * N)))
            hkvd = select_topk(dev0, k0)
            hkvd_per_layer.append(hkvd)

            # -- LAYERS 1 to L-1 (Gradual Filtering) --
            for i in range(1, self.L):
                target_k = k_targets[i-1]

                cached_K_hkvd = KV_new[i, 0, hkvd].clone()
                cached_V_hkvd = KV_new[i, 1, hkvd].clone()

                if is_llama:
                    fresh_K_hkvd, fresh_V_hkvd = run_layer_selective_llama_inplace(
                        self.layers[i], hidden, hkvd, KV_new[i], position_ids,
                        self.n_heads, self.n_kv_heads, self.head_dim, self.rotary_emb
                    )
                else:
                    fresh_K_hkvd, fresh_V_hkvd = run_layer_selective_gpt2_inplace(
                        self.layers[i], hidden, hkvd, KV_new[i]
                    )

                dev_i = compute_deviation_l2(
                    fresh_K_hkvd, fresh_V_hkvd, cached_K_hkvd, cached_V_hkvd, ~hit_mask[hkvd]
                )

                local_top = select_topk(dev_i, min(len(hkvd), target_k))
                hkvd = hkvd[local_top]
                hkvd_per_layer.append(hkvd)

        return KV_new, hkvd_per_layer

    def get_stats(self, hkvd_per_layer: List[torch.Tensor], N: int) -> dict:
        counts = [len(h) for h in hkvd_per_layer]
        corrected_ops = N + sum(counts[1:])
        full_ops = N * self.L
        return {
            "N"                  : N,
            "L"                  : self.L,
            "hkvd_counts"        : counts,
            "hkvd_ratios"        : [c / N for c in counts],
            "layer0_ratio"       : counts[0] / N if counts else 0,
            "final_layer_ratio"  : counts[-1] / N if counts else 0,
            "target_r"           : self.r,
            "corrected_total_ops": corrected_ops,
            "full_prefill_ops"   : full_ops,
            "true_savings_pct"   : (1 - corrected_ops / full_ops) * 100,
        }


# ─────────────────────────────────────────────────────────────────────────────
# Simple fixed-ratio selector (used by eval_harness for Table 1/2 sweep)
# ─────────────────────────────────────────────────────────────────────────────

class TokenSelector:
    """
    Fixed-ratio token selector.  Picks k = ceil(k_ratio * N) evenly-spaced
    token positions and returns them as a sorted int64 index tensor.

    Interface matches AdaptiveTokenSelector.select so both classes are
    drop-in interchangeable for eval_harness.py.
    """

    def __init__(self, k_ratio: float = 0.15):
        if not (0.0 < k_ratio <= 1.0):
            raise ValueError(f"k_ratio must be in (0, 1], got {k_ratio}")
        self.k_ratio = float(k_ratio)

    def select(self, chunk_ids: torch.Tensor, cached_kv: torch.Tensor) -> torch.Tensor:
        """
        Args:
            chunk_ids : (1, N) token IDs
            cached_kv : (L, 2, N, H, D)  — accepted but unused; kept for API parity
        Returns:
            Sorted int64 index tensor of length k on the same device as chunk_ids.
        """
        N = int(chunk_ids.shape[1])
        k = max(1, min(N, int(self.k_ratio * N)))
        # Evenly spaced positions cover the full sequence range
        indices = torch.linspace(0, N - 1, k, dtype=torch.int64, device=chunk_ids.device)
        return torch.sort(indices).values



In [ ]:
%%writefile cacheblendplus/blend_kernel.py
import os
import torch
from torch.utils.cpp_extension import load

# JIT compile the CUDA extension
_module = None
_load_error = None


def _load_cuda_module():
    """
    Best-effort CUDA extension loading.

    This is intentionally environment-agnostic:
    - no hardcoded CUDA/MSVC paths
    - respects user/system toolchain configuration
    - clean fallback to PyTorch scatter when build tools are unavailable
    """
    curr_dir = os.path.dirname(os.path.abspath(__file__))
    cuda_src = os.path.join(curr_dir, "blend.cu")

    if not os.path.exists(cuda_src):
        return None, "blend.cu not found"

    if not torch.cuda.is_available():
        return None, "CUDA is not available"

    # Allow users/CI to opt out of JIT compilation.
    if os.environ.get("CACHEBLEND_DISABLE_CUDA_EXT", "").lower() in {"1", "true", "yes"}:
        return None, "CUDA extension disabled by CACHEBLEND_DISABLE_CUDA_EXT"

    try:
        mod = load(
            name="blend_cuda",
            sources=[cuda_src],
            verbose=os.environ.get("CACHEBLEND_VERBOSE_EXT", "0") == "1",
        )
        return mod, None
    except Exception as exc:
        return None, str(exc)


_module, _load_error = _load_cuda_module()
if _module is None:
    print(f"Failed to load CUDA blend kernel: {_load_error}")
    print("Falling back to PyTorch implementation.")

def blend(cached_kv, new_values, indices):
    """
    Blends newly computed KV values into the cached KV tensor at the specified indices.
    
    Args:
        cached_kv: Tensor of shape (L, 2, N, H, D) - The baseline KV cache.
        new_values: Tensor of shape (L, 2, k, H, D) - The recomputed KV values.
        indices: Tensor of shape (k,) - The token indices that were recomputed.
        
    Returns:
        The updated cached_kv tensor.
    """
    if _module is not None and cached_kv.is_cuda and new_values.is_cuda and indices.is_cuda:
        # Use the optimized CUDA kernel
        # Note: the kernel modifies cached_kv in-place
        _module.launch_blend(cached_kv, new_values, indices)
        return cached_kv
    else:
        # Fallback to PyTorch scatter update
        # In-place scatter update across the token dimension (dim=2)
        cached_kv[:, :, indices, :, :] = new_values
        return cached_kv



In [ ]:
%%writefile cacheblendplus/pipeline.py
import time
import torch
from .kv_store import pack_kv, unpack_kv
from .token_selector import CacheBlendFusor

from .blend_kernel import blend

class KVBlender:
    def blend(self, cached_kv, new_values, indices):
        return blend(cached_kv, new_values, indices)

def cacheblend_generate(
    prompt,
    chunk_texts,
    model,
    tokenizer,
    store,
    selector=None, # For backward compatibility in eval_harness
    recomputer=None,
    blender=None,
    mode: str = "cacheblend",  # "cacheblend" or "standard_cache"
    max_new_tokens=64,
    min_new_tokens=8,
    do_sample=True,
    temperature=0.8,
    top_p=0.95,
    benchmark_first_token: bool = False,
    k_ratio: float = 0.15,
):
    """
    Refactored to use CacheBlendFusor for efficient, paper-faithful recomputation.
    """
    cache_hits = 0
    cache_misses = 0
    
    # 1. Gather/Prefill all chunks
    fused_kv_parts = []
    hit_mask_parts = []
    chunk_offsets = []
    current_offset = 0

    if torch.cuda.is_available():
        torch.cuda.synchronize()
    t_request_start = time.perf_counter()

    for chunk_text in chunk_texts:
        chunk_enc = tokenizer(
            chunk_text,
            return_tensors="pt",
            truncation=True,
            max_length=512,
        )
        chunk_ids = chunk_enc["input_ids"].cuda()
        N_c = chunk_ids.shape[1]
        
        cached = store.load(chunk_text, device="cuda")
        
        if cached is None:
            cache_misses += 1
            with torch.no_grad():
                out = model(chunk_ids, use_cache=True)
            chunk_kv = pack_kv(out.past_key_values)
            store.store(chunk_text, chunk_kv)
            # A miss means we computed it "fresh" in context 0, but since we just 
            # computed it, it's technically a "hit" for the fusion logic 
            # (or we treat it as something that needs recompute if we want to be safe)
            # Paper: misses are recomputed fully.
            hit_mask_parts.append(torch.zeros(N_c, dtype=torch.bool, device="cuda"))
        else:
            cache_hits += 1
            chunk_kv = cached
            hit_mask_parts.append(torch.ones(N_c, dtype=torch.bool, device="cuda"))

        fused_kv_parts.append(chunk_kv)
        chunk_offsets.append(current_offset)
        current_offset += N_c

    fused_kv = torch.cat(fused_kv_parts, dim=2)
    hit_mask = torch.cat(hit_mask_parts, dim=0)
    
    # Full prompt construction
    all_chunks_text = " ".join(chunk_texts)
    full_prompt_text = all_chunks_text + " " + prompt
    full_ids = tokenizer(full_prompt_text, return_tensors="pt").input_ids.cuda()
    
    # 2. Fuse and Recompute using the optimized Fusor
    if mode == "cacheblend":
        # If selector is an AdaptiveSelector, we can use its k_ratio
        r = k_ratio
        if hasattr(selector, "k_ratio"):
            r = selector.k_ratio
        elif hasattr(selector, "base_k_ratio"):
            r = selector.base_k_ratio
            
        fusor = CacheBlendFusor(model, r=r)
        # Note: fused_kv is updated in-place by the fusor
        fused_kv, _ = fusor.fuse(
            full_ids[:, :current_offset], 
            fused_kv, 
            hit_mask, 
            chunk_offsets=chunk_offsets
        )
    
    # 3. Final Decode
    prompt_ids = full_ids[:, current_offset:]
    past = unpack_kv(fused_kv)

    if torch.cuda.is_available():
        torch.cuda.synchronize()
    t_decode_start = time.perf_counter()

    context_len = fused_kv.shape[2]
    cache_position = torch.arange(
        context_len,
        context_len + prompt_ids.shape[1],
        device=prompt_ids.device
    )

    gen_kwargs = {
        "past_key_values": past,
        "cache_position": cache_position,
        "max_new_tokens": 1 if benchmark_first_token else max_new_tokens,
        "min_new_tokens": 1 if benchmark_first_token else min_new_tokens,
        "use_cache": True,
        "do_sample": False if benchmark_first_token else do_sample,
        "pad_token_id": tokenizer.eos_token_id,
        "eos_token_id": tokenizer.eos_token_id,
    }
    if gen_kwargs["do_sample"]:
        gen_kwargs["temperature"] = temperature
        gen_kwargs["top_p"] = top_p

    with torch.no_grad():
        out_ids = model.generate(prompt_ids, **gen_kwargs)

    if torch.cuda.is_available():
        torch.cuda.synchronize()
    ttft_ms = (time.perf_counter() - t_request_start) * 1000
    
    generated = out_ids[0]
    text = tokenizer.decode(generated, skip_special_tokens=True).strip()

    return {
        "text": text,
        "ttft_ms": ttft_ms,
        "cache_hits": cache_hits,
        "cache_misses": cache_misses,
    }



In [ ]:
%%writefile cacheblendplus/recompute_engine.py
"""
dependency:pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu124

Interface:
  recompute(chunk_ids, cached_kv, indices) → Tensor[L, 2, k, H, D]  float16, CUDA
"""

import torch
from .kv_store import pack_kv, unpack_kv 

class SelectiveRecomputer:
    """
    Runs a minimal forward pass over k selected tokens, using the full
    cached KV as past_key_values context so attention is computed correctly
    over the entire chunk and not just the k selected tokens.
    """

    def __init__(self, model):
        self.model = model

    def recompute(
        self,
        chunk_ids: torch.Tensor,   # (1, N)  full chunk token IDs
        cached_kv: torch.Tensor,   # (L, 2, N, H, D)  pre-computed KV
        indices: torch.Tensor,     # (k,)  int64, sorted ascending — from TokenSelector (Shubham's module)
    ) -> torch.Tensor:
        """
        Returns the freshly-computed KV entries for only the k selected tokens.
        Shape: (L, 2, k, H, D)

        The caller (pipeline.py) is responsible for scattering
        these back into the full cached_kv at positions `indices`.
        """
        assert indices.dtype == torch.int64, "indices must be int64"
        assert cached_kv.dtype == torch.float16, "cached_kv must be float16"
        assert cached_kv.device.type == "cuda", "cached_kv must be on CUDA"

        # Extract just the selected token IDs
        selected_ids = chunk_ids[:, indices]  # (1, k)

        # Unpack to HuggingFace tuple format so we can pass as past_key_values as this makes the model attend over the FULL cached context, not just k tokens
        past = unpack_kv(cached_kv) #added to fix transformers 4.45+ compatibility, which switched to DynamicCache for past_key_values instead of tuples

        with torch.no_grad():
            out = self.model(
                selected_ids,
                past_key_values=past,
                use_cache=True,
            )

        # HuggingFace appends the new k tokens at the END of past_key_values
        # We only want those fresh k entries, not the full N+k tensor
        new_kv_full = pack_kv(out.past_key_values)  # (L, 2, N+k, H, D)
        k = indices.shape[0]
        new_kv = new_kv_full[:, :, -k:, :, :]       # (L, 2, k, H, D)

        return new_kv


# ---------------------------------------------------------------------------
# Quick smoke test:
# Requires transformers, torch, kv_store.py (Adi's) in the same dir
# ---------------------------------------------------------------------------
if __name__ == "__main__":
    from transformers import AutoModelForCausalLM, AutoTokenizer
    import time

    MODEL_ID = "gpt2"
    print(f"Loading {MODEL_ID}...")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID, torch_dtype=torch.float16
    ).cuda().eval()

    text = "The quick brown fox jumps over the lazy dog. " * 4
    inputs = tokenizer(text, return_tensors="pt").to("cuda")
    chunk_ids = inputs["input_ids"]  # (1, N)
    N = chunk_ids.shape[1]
    print(f"Chunk length: {N} tokens")

    # Build a fake cached KV by running a real forward pass
    with torch.no_grad():
        out = model(chunk_ids, use_cache=True)
    cached_kv = pack_kv(out.past_key_values)
    print(f"cached_kv shape: {cached_kv.shape}  (L, 2, N, H, D)")

    # Pick ~15% of tokens as "high divergence" (random stand-in for the selector)
    k = max(1, int(0.15 * N))
    indices = torch.randint(0, N, (k,)).unique().sort().values.to(torch.int64).cuda()
    print(f"Recomputing {len(indices)}/{N} tokens ({len(indices)/N:.0%})")

    recomputer = SelectiveRecomputer(model)

    t0 = time.perf_counter()
    new_kv = recomputer.recompute(chunk_ids, cached_kv, indices)
    elapsed = (time.perf_counter() - t0) * 1000

    print(f"new_kv shape: {new_kv.shape}  (L, 2, k, H, D)")
    print(f"Recompute time: {elapsed:.1f} ms")
    assert new_kv.shape[2] == len(indices), "k dimension mismatch!"
    assert new_kv.dtype == torch.float16
    print("✓ Smoke test passed")



In [ ]:
%%writefile cacheblendplus/adaptive_selector.py
import torch
import torch.nn.functional as F
from typing import Dict, List


class AdaptiveTokenSelector:
    """
    Paper-aligned adaptive selector:
    1) Compute token divergence from layer-0 fresh hidden states.
    2) Adapt recompute ratio from mean divergence.
    3) Return top-k divergent token indices (sorted ascending, int64).

    This is a drop-in selector for pipeline.py:
        indices = selector.select(chunk_ids, cached_kv)
    """

    def __init__(
        self,
        model,
        base_k_ratio: float = 0.15,
        low_thresh: float = 0.05,
        high_thresh: float = 0.20,
        min_k_ratio: float = 0.05,
        max_k_ratio: float = 0.30,
        eps: float = 1e-8,
        require_cuda: bool = True,
    ):
        self.model = model
        self.base_k_ratio = float(base_k_ratio)
        self.low_thresh = float(low_thresh)
        self.high_thresh = float(high_thresh)
        self.min_k_ratio = float(min_k_ratio)
        self.max_k_ratio = float(max_k_ratio)
        self.eps = float(eps)
        self.require_cuda = bool(require_cuda)

        if not (0.0 < self.min_k_ratio <= self.max_k_ratio <= 1.0):
            raise ValueError("Expected 0 < min_k_ratio <= max_k_ratio <= 1.")
        if not (0.0 <= self.low_thresh <= self.high_thresh):
            raise ValueError("Expected 0 <= low_thresh <= high_thresh.")
        if not (0.0 < self.base_k_ratio <= 1.0):
            raise ValueError("Expected 0 < base_k_ratio <= 1.")

        self.model.eval()
        self.history: List[Dict[str, float]] = []
        self.last_selection: Dict[str, float] = {}

    def _project_cached_for_cosine(self, cached_k_mean: torch.Tensor, target_dim: int) -> torch.Tensor:
        """
        Align cached representation dim to model hidden size for cosine.
        Uses deterministic pad/truncate to avoid learnable dependencies.
        """
        cached_dim = cached_k_mean.shape[-1]
        if cached_dim == target_dim:
            return cached_k_mean
        if cached_dim > target_dim:
            return cached_k_mean[:, :target_dim]

        pad = target_dim - cached_dim
        return F.pad(cached_k_mean, (0, pad), mode="constant", value=0.0)

    def _adaptive_ratio(self, mean_divergence: float) -> float:
        if mean_divergence <= self.low_thresh:
            return self.min_k_ratio
        if mean_divergence >= self.high_thresh:
            return self.max_k_ratio

        # Linear interpolation in the transition band.
        span = max(self.high_thresh - self.low_thresh, self.eps)
        alpha = (mean_divergence - self.low_thresh) / span
        return self.min_k_ratio + alpha * (self.max_k_ratio - self.min_k_ratio)

    def select(self, chunk_ids: torch.Tensor, cached_kv: torch.Tensor) -> torch.Tensor:
        """
        Args:
            chunk_ids: (1, N) token IDs
            cached_kv: (L, 2, N, H, D) cache tensor
        Returns:
            Tensor[k] int64, sorted ascending
        """
        if chunk_ids.ndim != 2 or chunk_ids.shape[0] != 1:
            raise ValueError(f"Expected chunk_ids shape (1, N), got {tuple(chunk_ids.shape)}")
        if cached_kv.ndim != 5 or cached_kv.shape[1] != 2:
            raise ValueError(f"Expected cached_kv shape (L, 2, N, H, D), got {tuple(cached_kv.shape)}")
        if self.require_cuda and (chunk_ids.device.type != "cuda" or cached_kv.device.type != "cuda"):
            raise ValueError("AdaptiveTokenSelector requires CUDA tensors.")

        n_tokens = int(chunk_ids.shape[1])
        if int(cached_kv.shape[2]) != n_tokens:
            raise ValueError("chunk_ids length must match cached_kv token dimension.")

        with torch.no_grad():
            # We use an all-ones mask because chunk inputs here are unpadded.
            attn_mask = torch.ones_like(chunk_ids, device=chunk_ids.device)
            out = self.model(
                chunk_ids,
                attention_mask=attn_mask,
                use_cache=True,
                output_hidden_states=True,
                return_dict=True,
            )
            # hidden_states[0] is embedding output; hidden_states[1] is post layer-0.
            fresh_hidden = out.hidden_states[1].squeeze(0).float()  # (N, d_model)

        cached_k = cached_kv[0, 0].float()  # (N, H, D)
        cached_hidden = cached_k.mean(dim=1)  # (N, D)
        cached_hidden = self._project_cached_for_cosine(cached_hidden, fresh_hidden.shape[-1])

        fresh_norm = F.normalize(fresh_hidden + self.eps, p=2, dim=-1)
        cached_norm = F.normalize(cached_hidden + self.eps, p=2, dim=-1)

        cosine = F.cosine_similarity(fresh_norm, cached_norm, dim=-1)
        divergence = (1.0 - cosine).clamp_min(0.0)

        mean_div = float(divergence.mean().item())
        adaptive_ratio = self._adaptive_ratio(mean_div)

        k = max(1, int(adaptive_ratio * n_tokens))
        k = min(k, n_tokens)
        indices = torch.topk(divergence, k, largest=True, sorted=False).indices
        indices = torch.sort(indices).values.to(dtype=torch.int64)

        self.last_selection = {
            "sequence_length": float(n_tokens),
            "mean_divergence": mean_div,
            "selected_k_ratio": float(adaptive_ratio),
            "selected_k": float(k),
            "base_k_ratio": float(self.base_k_ratio),
        }
        self.history.append(self.last_selection.copy())

        return indices

    def get_last_selection_stats(self) -> Dict[str, float]:
        return self.last_selection.copy()

    def get_selection_history(self) -> List[Dict[str, float]]:
        return list(self.history)

    def reset_history(self) -> None:
        self.history.clear()
        self.last_selection = {}


# Alias for simpler imports in scripts/notebooks.
AdaptiveSelector = AdaptiveTokenSelector



In [ ]:
import sys, os
sys.path.append(os.getcwd())
from cacheblendplus.kv_store import pack_kv, unpack_kv, KVCacheStore
from cacheblendplus.token_selector import TokenSelector, CacheBlendFusor
from cacheblendplus.recompute_engine import SelectiveRecomputer
from cacheblendplus.pipeline import cacheblend_generate, KVBlender
from cacheblendplus.adaptive_selector import AdaptiveSelector


In [ ]:
# Runtime + model load
assert torch.cuda.is_available(), "Please enable GPU runtime in Colab (T4)."

device = "cuda"
# Use GPT-2 for cleaner behavior than tiny-gpt2 while still fitting Colab T4 comfortably.
model_id = "gpt2"

print(f"Loading: {model_id}")
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(model_id, dtype=torch.float16).to(device).eval()

print("Model loaded.")

In [ ]:
# Direct Colab smoke tests + repeated benchmark (no pytest needed)

# Test 1: adaptive selector output contract
text = (
    "CacheBlend recomputes only highly divergent tokens to reduce prefill latency. "
    "This validates adaptive token selection behavior."
)
chunk_enc = tokenizer(text, return_tensors="pt", return_attention_mask=True)
chunk_ids = chunk_enc["input_ids"].to(device)
chunk_mask = chunk_enc["attention_mask"].to(device)

with torch.no_grad():
    out = model(chunk_ids, attention_mask=chunk_mask, use_cache=True)

cached_kv = pack_kv(out.past_key_values).to(device)
selector = AdaptiveTokenSelector(model=model)
indices = selector.select(chunk_ids, cached_kv)
stats = selector.last_selection

assert indices.dtype == torch.int64
assert indices.device.type == "cuda"
assert indices.numel() >= 1
assert torch.all(indices[1:] >= indices[:-1]), "Indices must be sorted"
assert 0.05 <= stats["selected_k_ratio"] <= 0.30
print("Test 1 passed:", stats)


# Test 2: end-to-end cold/warm run
store = KVCacheStore()
recomputer = SelectiveRecomputer(model)
blender = KVBlender()

chunks = [
    "Paris is the capital of France and has many famous landmarks. " * 8,
    "The Eiffel Tower is one of the most visited monuments in the world. " * 8,
    "CacheBlend reuses cached KV and selectively recomputes high-divergence tokens. " * 8,
    "Selective recomputation reduces full prefill cost when cached context is reusable. " * 8,
]
prompt = "Summarize how CacheBlend can reduce latency for repeated context queries."

cold = cacheblend_generate(
    prompt, chunks, model, tokenizer, store, selector, recomputer, blender, max_new_tokens=24
)
warm = cacheblend_generate(
    prompt, chunks, model, tokenizer, store, selector, recomputer, blender, max_new_tokens=24
)

assert cold["cache_misses"] == len(chunks)
assert warm["cache_hits"] == len(chunks)
assert len(warm["text"]) > 0
assert warm["ttft_ms"] > 0

print("Test 2 passed")
print(f"COLD TTFT: {cold['ttft_ms']:.2f} ms")
print(f"WARM TTFT: {warm['ttft_ms']:.2f} ms")
print(f"Speedup: {cold['ttft_ms']/warm['ttft_ms']:.2f}x")
print("Output:", warm["text"][:200])
print("Output repr:", repr(warm["text"][:200]))


# Test 3: repeated benchmark for stable report numbers
bench = run_three_mode_benchmark(
    prompt=prompt,
    chunk_texts=chunks,
    model=model,
    tokenizer=tokenizer,
    trials=5,
    warmup_runs=1,
)

print("\nBenchmark summary:")
for k, v in bench["summary"].items():
    if isinstance(v, float):
        print(f"  {k}: {v:.4f}")
    else:
        print(f"  {k}: {v}")

with open("cacheblend_colab_results.json", "w") as f:
    json.dump(bench, f, indent=2)

print("Saved benchmark JSON -> cacheblend_colab_results.json")

## Phase 2 — Full Eval Harness (Mistral-7B · MultiNews)

Produces the three tables from the CacheBlend+ report:

| Table | Metric | Configs |
|-------|--------|---------|
| 1 | TTFT (ms) | k/N ∈ {5 %, 10 %, 15 %, 20 %, 30 %} vs full-prefill |
| 2 | ROUGE-L | same k/N sweep |
| 3 | ROUGE-L grid | {fixed, adaptive} × {exact, semantic} |

> **GPU required.** Run on a Colab A100 or T4 instance.  
> Results are saved incrementally to `/content/drive/MyDrive/cacheblend_results/`  
> so the notebook is safe to preempt and resume.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
RESULTS_DIR = '/content/drive/MyDrive/cacheblend_results'
os.makedirs(RESULTS_DIR, exist_ok=True)
print(f'Results will be saved to: {RESULTS_DIR}')


In [ ]:
"""
eval_harness.py — Pranav's Phase 2-3 deliverable
Produces three tables for the final report:

  Table 1: TTFT (ms)   at k/N in {0.05, 0.10, 0.15, 0.20, 0.30} vs full-prefill
  Table 2: ROUGE-L     at same k/N values
  Table 3: 2x2 grid    {adaptive, fixed} x {semantic, exact}
           (semantic column is N/A until Adithya's module is ready)

Dataset: MultiNews (HuggingFace) — 60 samples, matching paper's evaluation setup.

Usage on Colab:
  !pip install evaluate rouge_score datasets -q
  %run eval_harness.py --n_samples 60 --output_dir /content/drive/MyDrive/results

All results saved to JSON incrementally — safe to preempt and resume.
"""

import argparse
import json
import os
import sys
import time
import torch
from pathlib import Path

# Make the package importable when run as a script from scripts/ or via %run
_repo_root = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))
if _repo_root not in sys.path:
    sys.path.insert(0, _repo_root)

from cacheblendplus.pipeline import cacheblend_generate, KVBlender
from cacheblendplus.kv_store import KVCacheStore
from cacheblendplus.recompute_engine import SelectiveRecomputer
from cacheblendplus.token_selector import TokenSelector


# ---------------------------------------------------------------------------
# ROUGE-L scorer
# ---------------------------------------------------------------------------
def load_rouge():
    try:
        import evaluate
        return evaluate.load("rouge")
    except Exception:
        print("WARNING: 'evaluate' not installed. Run: pip install evaluate rouge_score")
        return None


# ---------------------------------------------------------------------------
# Dataset loader — MultiNews
# ---------------------------------------------------------------------------
def load_multinews(n_samples: int) -> list:
    try:
        from datasets import load_dataset
        ds = load_dataset("multi_news", split=f"validation[:{n_samples}]",
                          trust_remote_code=True)
        samples = []
        for ex in ds:
            articles = [a.strip() for a in ex["document"].split("|||||")
                        if a.strip()]
            summary = ex["summary"].strip()
            if articles and summary:
                samples.append({
                    "chunks": articles[:4],  # cap at 4 chunks like the paper
                    "summary": summary,
                })
        return samples[:n_samples]
    except Exception as e:
        print(f"WARNING: Could not load MultiNews ({e}). Using fallback.")
        return [{
            "chunks": [
                "Scientists discovered a new deep-sea fish species in the Pacific Ocean. "
                "The fish lives at depths over 3000 meters and has bioluminescent properties.",
                "The discovery adds to a growing list of deep-sea species. "
                "The fish has been named Abyssus luminis by researchers.",
            ],
            "summary": "Researchers found a new bioluminescent deep-sea fish in the Pacific.",
        }] * min(n_samples, 5)


# ---------------------------------------------------------------------------
# TTFT measurement — prefill only, matches paper's definition
# ---------------------------------------------------------------------------
def measure_prefill_ttft(model, input_ids: torch.Tensor, n_runs: int = 3) -> float:
    times = []
    for _ in range(n_runs):
        torch.cuda.synchronize()
        t0 = time.perf_counter()
        with torch.no_grad():
            model(input_ids, use_cache=True)
        torch.cuda.synchronize()
        times.append((time.perf_counter() - t0) * 1000)
    return sorted(times)[n_runs // 2]


# ---------------------------------------------------------------------------
# Generation for ROUGE-L measurement
# ---------------------------------------------------------------------------
def generate_summary(model, tokenizer, chunks: list, max_new_tokens: int = 128) -> str:
    context = " ".join(chunks)
    prompt = f"{context}\n\nSummarize the above in 2-3 sentences:"
    inputs = tokenizer(
        prompt, return_tensors="pt", truncation=True, max_length=2048
    ).to("cuda")
    context_len = inputs["input_ids"].shape[1]
    with torch.no_grad():
        out = model.generate(
            inputs["input_ids"],
            max_new_tokens=max_new_tokens,
            use_cache=True,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(out[0][context_len:], skip_special_tokens=True)


# ---------------------------------------------------------------------------
# Table 1 + 2
# ---------------------------------------------------------------------------
def run_table1_table2(
    model, tokenizer, samples, recompute_ratios,
    selector_class, store_class, recomputer, blender,
    output_dir, max_new_tokens=128
) -> dict:

    outpath = Path(output_dir) / "table1_table2.json"
    results = {}
    if outpath.exists():
        with open(outpath) as f:
            results = json.load(f)
        print(f"Resuming from {outpath} ({len(results)} entries already done)")

    rouge = load_rouge()

    # --- Baseline ---
    if "baseline" not in results:
        print("\n[Baseline] Full prefill TTFT + generation quality...")
        base_ttfts, base_preds = [], []
        for i, s in enumerate(samples):
            all_text = " ".join(s["chunks"])
            ids = tokenizer(all_text, return_tensors="pt",
                            truncation=True, max_length=2048)["input_ids"].cuda()
            base_ttfts.append(measure_prefill_ttft(model, ids))
            base_preds.append(
                generate_summary(model, tokenizer, s["chunks"], max_new_tokens)
            )
            if (i + 1) % 10 == 0:
                print(f"  {i+1}/{len(samples)}")

        base_rouge = None
        if rouge:
            refs = [s["summary"] for s in samples]
            base_rouge = rouge.compute(predictions=base_preds,
                                       references=refs)["rougeL"]

        results["baseline"] = {
            "mean_ttft_ms": sum(base_ttfts) / len(base_ttfts),
            "rougeL": base_rouge,
        }
        with open(outpath, "w") as f:
            json.dump(results, f, indent=2)
        rl = results["baseline"]["rougeL"]
        print(f"  Baseline: TTFT={results['baseline']['mean_ttft_ms']:.1f}ms "
              f"| ROUGE-L={f'{rl:.4f}' if rl else 'N/A'}")

    # --- CacheBlend sweep ---
    for ratio in recompute_ratios:
        key = f"ratio_{ratio:.2f}"
        if key in results:
            print(f"  Skipping ratio={ratio} (cached)")
            continue

        print(f"\n[CacheBlend] ratio={ratio:.0%}...")
        store    = store_class()
        selector = selector_class(k_ratio=ratio)
        cb_ttfts, cb_preds = [], []

        for i, s in enumerate(samples):
            chunks = s["chunks"]
            prompt = "Summarize the above in 2-3 sentences:"

            # Cold call to populate cache
            cacheblend_generate(prompt, chunks, model, tokenizer,
                                store, selector, recomputer, blender,
                                max_new_tokens=1)

            # TTFT: prefill on k% of tokens (warm path equivalent)
            all_text = " ".join(chunks)
            ids = tokenizer(all_text, return_tensors="pt",
                            truncation=True, max_length=2048)["input_ids"].cuda()
            k = max(1, int(ratio * ids.shape[1]))
            cb_ttfts.append(measure_prefill_ttft(model, ids[:, :k]))

            # Quality: warm generation
            result = cacheblend_generate(prompt, chunks, model, tokenizer,
                                         store, selector, recomputer, blender,
                                         max_new_tokens=max_new_tokens)
            cb_preds.append(result["text"])

            if (i + 1) % 10 == 0:
                print(f"  {i+1}/{len(samples)}")

        cb_rouge = None
        if rouge:
            refs = [s["summary"] for s in samples]
            cb_rouge = rouge.compute(predictions=cb_preds,
                                     references=refs)["rougeL"]

        mean_ttft = sum(cb_ttfts) / len(cb_ttfts)
        results[key] = {
            "ratio": ratio,
            "mean_ttft_ms": mean_ttft,
            "speedup_vs_baseline": results["baseline"]["mean_ttft_ms"] / mean_ttft,
            "rougeL": cb_rouge,
        }
        with open(outpath, "w") as f:
            json.dump(results, f, indent=2)
        rl = cb_rouge
        print(f"  ratio={ratio:.0%}: TTFT={mean_ttft:.1f}ms "
              f"| speedup={results[key]['speedup_vs_baseline']:.2f}x "
              f"| ROUGE-L={f'{rl:.4f}' if rl else 'N/A'}")

    return results


# ---------------------------------------------------------------------------
# Table 3: {adaptive, fixed} x {exact, semantic}
# ---------------------------------------------------------------------------
def run_table3(
    model, tokenizer, samples,
    fixed_selector_class, adaptive_selector_class,
    exact_store_class, semantic_store_class,
    recomputer, blender,
    output_dir, k_ratio=0.15, max_new_tokens=128
) -> dict:

    outpath = Path(output_dir) / "table3.json"
    results = {}
    if outpath.exists():
        with open(outpath) as f:
            results = json.load(f)

    rouge = load_rouge()

    configs = [("fixed", "exact", fixed_selector_class, exact_store_class)]
    if adaptive_selector_class is not None:
        configs.append(("adaptive", "exact", adaptive_selector_class, exact_store_class))
    if semantic_store_class is not None:
        configs.append(("fixed",    "semantic", fixed_selector_class,    semantic_store_class))
        configs.append(("adaptive", "semantic", adaptive_selector_class, semantic_store_class))

    for sel_type, store_type, sel_cls, store_cls in configs:
        key = f"{sel_type}_{store_type}"
        if key in results:
            print(f"  Skipping {key}")
            continue

        print(f"\n[Table 3] {sel_type} x {store_type}...")
        store    = store_cls()
        selector = sel_cls(k_ratio=k_ratio)
        preds    = []

        for i, s in enumerate(samples):
            chunks = s["chunks"]
            prompt = "Summarize the above in 2-3 sentences:"
            cacheblend_generate(prompt, chunks, model, tokenizer,
                                store, selector, recomputer, blender,
                                max_new_tokens=1)
            result = cacheblend_generate(prompt, chunks, model, tokenizer,
                                         store, selector, recomputer, blender,
                                         max_new_tokens=max_new_tokens)
            preds.append(result["text"])
            if (i + 1) % 10 == 0:
                print(f"  {i+1}/{len(samples)}")

        score = None
        if rouge:
            refs = [s["summary"] for s in samples]
            score = rouge.compute(predictions=preds, references=refs)["rougeL"]

        results[key] = {"selector": sel_type, "store": store_type, "rougeL": score}
        with open(outpath, "w") as f:
            json.dump(results, f, indent=2)
        print(f"  {key}: ROUGE-L={f'{score:.4f}' if score else 'N/A'}")

    return results


# ---------------------------------------------------------------------------
# Print tables
# ---------------------------------------------------------------------------
def print_tables(t12: dict, t3: dict):
    print("\n" + "=" * 65)
    print("TABLE 1 & 2  |  MultiNews  |  Mistral-7B  |  A100")
    print("=" * 65)
    b = t12.get("baseline", {})
    print(f"{'Config':<22} {'TTFT (ms)':>10} {'Speedup':>10} {'ROUGE-L':>10}")
    print("-" * 54)
    bl = b.get("rougeL")
    print(f"{'Full prefill':<22} {b.get('mean_ttft_ms', 0):>10.1f} "
          f"{'1.00x':>10} {f'{bl:.4f}' if bl else 'N/A':>10}")
    for k in sorted(t12):
        if not k.startswith("ratio_"):
            continue
        v = t12[k]
        rl = v.get("rougeL")
        label = f"CacheBlend {int(v['ratio']*100)}%"
        print(f"{label:<22} {v['mean_ttft_ms']:>10.1f} "
              f"{v['speedup_vs_baseline']:>9.2f}x "
              f"{f'{rl:.4f}' if rl else 'N/A':>10}")

    if t3:
        print("\n" + "=" * 45)
        print("TABLE 3  |  ROUGE-L by selector x store")
        print("=" * 45)
        print(f"{'':>12} {'Exact':>12} {'Semantic':>12}")
        print("-" * 38)
        for sel in ["fixed", "adaptive"]:
            exact    = t3.get(f"{sel}_exact",    {}).get("rougeL")
            semantic = t3.get(f"{sel}_semantic", {}).get("rougeL")
            print(f"{sel.capitalize():<12} "
                  f"{f'{exact:.4f}' if exact else 'N/A':>12} "
                  f"{f'{semantic:.4f}' if semantic else 'N/A':>12}")


# ---------------------------------------------------------------------------
# Entry point
# ---------------------------------------------------------------------------
def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--model_id", default="mistralai/Mistral-7B-Instruct-v0.2")
    parser.add_argument("--recompute_ratios", nargs="+", type=float,
                        default=[0.05, 0.10, 0.15, 0.20, 0.30])
    parser.add_argument("--n_samples", type=int, default=60)
    parser.add_argument("--max_new_tokens", type=int, default=128)
    parser.add_argument("--output_dir", default="results")
    parser.add_argument("--skip_table3", action="store_true")
    args = parser.parse_args()

    os.makedirs(args.output_dir, exist_ok=True)

    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

    try:
        from cacheblendplus.adaptive_selector import AdaptiveSelector
        print("AdaptiveSelector loaded")
    except ImportError:
        print("WARNING: adaptive_selector not found — using TokenSelector for Table 3")
        AdaptiveSelector = None

    try:
        from cacheblendplus.semantic_kv_store import SemanticKVCacheStore
        print("SemanticKVCacheStore loaded")
    except ImportError:
        print("NOTE: semantic_kv_store not found — Table 3 semantic column = N/A")
        SemanticKVCacheStore = None

    print(f"Loading {args.model_id}...")
    bnb = BitsAndBytesConfig(load_in_8bit=True)
    tokenizer = AutoTokenizer.from_pretrained(args.model_id)
    tokenizer.pad_token = tokenizer.eos_token
    model = AutoModelForCausalLM.from_pretrained(
        args.model_id, quantization_config=bnb, device_map="cuda"
    ).eval()

    recomputer = SelectiveRecomputer(model)
    blender    = KVBlender()

    print(f"Loading {args.n_samples} MultiNews samples...")
    samples = load_multinews(args.n_samples)
    print(f"  Loaded {len(samples)} samples")

    t12 = run_table1_table2(
        model, tokenizer, samples, args.recompute_ratios,
        TokenSelector, KVCacheStore, recomputer, blender,
        args.output_dir, args.max_new_tokens,
    )

    t3 = {}
    if not args.skip_table3:
        # AdaptiveSelector needs model as first arg; wrap into a k_ratio-only factory
        # so run_table3 can call sel_cls(k_ratio=...) uniformly for all selector types.
        if AdaptiveSelector is not None:
            adaptive_factory = lambda k_ratio, _m=model: AdaptiveSelector(
                _m, base_k_ratio=k_ratio
            )
        else:
            adaptive_factory = None

        t3 = run_table3(
            model, tokenizer, samples,
            TokenSelector, adaptive_factory,
            KVCacheStore, SemanticKVCacheStore,
            recomputer, blender,
            args.output_dir, k_ratio=0.15,
            max_new_tokens=args.max_new_tokens,
        )

    print_tables(t12, t3)
    print(f"\nAll results saved to {args.output_dir}/")


if __name__ == "__main__":
    main()


In [ ]:
# Load Mistral-7B-Instruct in 8-bit (fits on T4/A100)
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MISTRAL_ID = 'mistralai/Mistral-7B-Instruct-v0.2'
print(f'Loading {MISTRAL_ID} in 8-bit ...')

bnb_cfg = BitsAndBytesConfig(load_in_8bit=True)
mistral_tok = AutoTokenizer.from_pretrained(MISTRAL_ID)
mistral_tok.pad_token = mistral_tok.eos_token
mistral_model = AutoModelForCausalLM.from_pretrained(
    MISTRAL_ID,
    quantization_config=bnb_cfg,
    device_map='cuda',
).eval()

mistral_recomputer = SelectiveRecomputer(mistral_model)
mistral_blender    = KVBlender()
print('Mistral-7B ready.')


In [ ]:
N_SAMPLES        = 60
MAX_NEW_TOKENS   = 128
RECOMPUTE_RATIOS = [0.05, 0.10, 0.15, 0.20, 0.30]

print(f'Loading {N_SAMPLES} MultiNews samples ...')
mn_samples = load_multinews(N_SAMPLES)
print(f'  Loaded {len(mn_samples)} samples')

t12_results = eval_table1_table2(
    mistral_model, mistral_tok, mn_samples,
    RECOMPUTE_RATIOS,
    mistral_recomputer, mistral_blender,
    RESULTS_DIR,
    max_new_tokens=MAX_NEW_TOKENS,
)


In [ ]:
# Sequence-length diagnostic — run after mn_samples is loaded
# Shows actual token counts so we can judge whether context is long enough
# for meaningful TTFT speedups (paper uses 2k-8k token contexts).
_lens = [
    len(mistral_tok(' '.join(s['chunks']), truncation=True,
                    max_length=4096)['input_ids'])
    for s in mn_samples
]
_sorted = sorted(_lens)
print(f'Sequence lengths across {len(_lens)} samples:')
print(f'  Mean : {sum(_lens)/len(_lens):.0f} tokens')
print(f'  Min  : {_sorted[0]} tokens')
print(f'  p25  : {_sorted[len(_sorted)//4]} tokens')
print(f'  p50  : {_sorted[len(_sorted)//2]} tokens')
print(f'  p75  : {_sorted[3*len(_sorted)//4]} tokens')
print(f'  Max  : {_sorted[-1]} tokens')
if sum(_lens)/len(_lens) < 600:
    print('NOTE: Mean sequence is short (<600 tokens). '
          'Speedups will be modest. Consider using more chunks per sample.')


In [ ]:
# sentence-transformers needed for semantic store (Table 3 col 2).
# Set USE_SEMANTIC=False to skip it.
USE_SEMANTIC = True

SemanticStore = None
if USE_SEMANTIC:
    try:
        from sentence_transformers import SentenceTransformer
        import numpy as _np

        class SemanticKVCacheStore(KVCacheStore):
            def __init__(self, model_name='all-MiniLM-L6-v2', threshold=0.85):
                super().__init__()
                self.encoder   = SentenceTransformer(model_name, device='cpu')
                self.threshold = threshold
                self._embeddings = []

            def store(self, text, kv_tensor):
                self._data[text] = kv_tensor.cpu().clone()
                emb = self.encoder.encode(text, convert_to_numpy=True)
                self._embeddings.append((emb, text))

            def load(self, text, device='cuda'):
                kv = self._data.get(text)
                if kv is not None:
                    return kv.to(device)
                if not self._embeddings:
                    return None
                q = self.encoder.encode(text, convert_to_numpy=True)
                best_sim, best_key = -1.0, None
                for emb, key in self._embeddings:
                    sim = float(_np.dot(q, emb) /
                                (_np.linalg.norm(q) * _np.linalg.norm(emb) + 1e-8))
                    if sim > best_sim:
                        best_sim, best_key = sim, key
                if best_sim >= self.threshold and best_key in self._data:
                    return self._data[best_key].to(device)
                return None

        SemanticStore = SemanticKVCacheStore
        print('SemanticKVCacheStore ready.')
    except ImportError:
        print('sentence-transformers not installed -- semantic column will be N/A')

t3_results = eval_table3(
    mistral_model, mistral_tok, mn_samples,
    KVCacheStore, SemanticStore,
    mistral_recomputer, mistral_blender,
    RESULTS_DIR, k_ratio=0.15,
    max_new_tokens=MAX_NEW_TOKENS,
)


### ROUGE-L Repair

The eval ran successfully but `evaluate` wasn't importable during that session,
so all `rougeL` fields are `null`. This cell re-generates predictions for every
null entry (skipping TTFT re-measurement) and fills in the scores.

In [ ]:
# ROUGE-L Repair: fills null rougeL without re-measuring TTFT
import json, importlib
from pathlib import Path

# Force-reload evaluate in case it was installed mid-session
try:
    import evaluate as _ev
    importlib.reload(_ev)
    rouge = _ev.load("rouge")
    print(f"evaluate {_ev.__version__} ready")
except Exception as e:
    raise RuntimeError(
        f"evaluate not importable: {e}\n"
        "Run:  !pip install evaluate rouge_score -q\n"
        "then Restart Runtime and re-run from the top."
    )


def _generate_preds(model, tokenizer, samples, store_class, sel_fn,
                    recomputer, blender, max_new_tokens=128):
    # Re-run warm cacheblend_generate for each sample; return list of prediction strings.
    preds = []
    for i, s in enumerate(samples):
        chunks = s["chunks"]
        prompt = "Summarize the above in 2-3 sentences:"
        store    = store_class()
        selector = sel_fn()
        # Cold pass to populate cache
        cacheblend_generate(prompt, chunks, model, tokenizer,
                            store, selector, recomputer, blender,
                            max_new_tokens=1, do_sample=False)
        # Warm pass for quality
        r = cacheblend_generate(prompt, chunks, model, tokenizer,
                                store, selector, recomputer, blender,
                                max_new_tokens=max_new_tokens, do_sample=False)
        preds.append(r["text"])
        if (i + 1) % 10 == 0:
            print(f"  {i+1}/{len(samples)}")
    return preds


def repair_rouge(results_dir, model, tokenizer, samples,
                 recomputer, blender, max_new_tokens=128):
    refs = [s["summary"] for s in samples]

    # ── table1_table2.json ────────────────────────────────────────────────
    t12_path = Path(results_dir) / "table1_table2.json"
    with open(t12_path) as f:
        t12 = json.load(f)

    t12_dirty = False

    # Baseline — uses standard_cache (no selector needed)
    if t12.get("baseline", {}).get("rougeL") is None and "baseline" in t12:
        print("\n[Repair] baseline ROUGE-L ...")
        preds = _generate_preds(
            model, tokenizer, samples,
            KVCacheStore,
            lambda: TokenSelector(k_ratio=1.0),   # k=100% => standard cache behaviour
            recomputer, blender, max_new_tokens,
        )
        t12["baseline"]["rougeL"] = rouge.compute(
            predictions=preds, references=refs
        )["rougeL"]
        t12_dirty = True
        print(f"  baseline rougeL = {t12['baseline']['rougeL']:.4f}")

    for key, entry in t12.items():
        if not key.startswith("ratio_") or entry.get("rougeL") is not None:
            continue
        ratio = entry["ratio"]
        print(f"\n[Repair] ratio={ratio:.0%} ROUGE-L ...")
        preds = _generate_preds(
            model, tokenizer, samples,
            KVCacheStore,
            lambda r=ratio: TokenSelector(k_ratio=r),
            recomputer, blender, max_new_tokens,
        )
        entry["rougeL"] = rouge.compute(
            predictions=preds, references=refs
        )["rougeL"]
        t12_dirty = True
        print(f"  ratio={ratio:.0%} rougeL = {entry['rougeL']:.4f}")

    if t12_dirty:
        with open(t12_path, "w") as f:
            json.dump(t12, f, indent=2)
        print(f"\nSaved updated {t12_path}")

    # ── table3.json ────────────────────────────────────────────────────────
    t3_path = Path(results_dir) / "table3.json"
    if not t3_path.exists():
        print("table3.json not found — skipping")
        return t12, {}

    with open(t3_path) as f:
        t3 = json.load(f)

    t3_dirty = False
    sel_map = {
        "fixed":    lambda: TokenSelector(k_ratio=0.15),
        "adaptive": lambda: AdaptiveTokenSelector(
            model=model,
            low_thresh=0.05, high_thresh=0.20,
            min_k_ratio=0.075, max_k_ratio=0.30,
        ),
    }

    for key, entry in t3.items():
        if entry.get("rougeL") is not None:
            continue
        sel_type   = entry["selector"]
        store_type = entry["store"]
        print(f"\n[Repair] Table 3  {sel_type} x {store_type} ...")

        if store_type == "semantic":
            try:
                from sentence_transformers import SentenceTransformer
                import numpy as _np

                class _SemanticStore(KVCacheStore):
                    def __init__(self):
                        super().__init__()
                        self.encoder   = SentenceTransformer("all-MiniLM-L6-v2", device="cpu")
                        self.threshold = 0.85
                        self._embeddings = []
                    def store(self, text, kv):
                        self._data[text] = kv.cpu().clone()
                        self._embeddings.append(
                            (self.encoder.encode(text, convert_to_numpy=True), text)
                        )
                    def load(self, text, device="cuda"):
                        kv = self._data.get(text)
                        if kv is not None:
                            return kv.to(device)
                        if not self._embeddings:
                            return None
                        q = self.encoder.encode(text, convert_to_numpy=True)
                        best_sim, best_key = -1.0, None
                        for emb, k in self._embeddings:
                            sim = float(_np.dot(q, emb) /
                                        (_np.linalg.norm(q)*_np.linalg.norm(emb)+1e-8))
                            if sim > best_sim:
                                best_sim, best_key = sim, k
                        if best_sim >= self.threshold and best_key in self._data:
                            return self._data[best_key].to(device)
                        return None

                store_cls = _SemanticStore
            except ImportError:
                print("  sentence-transformers not installed — skipping semantic entry")
                continue
        else:
            store_cls = KVCacheStore

        preds = _generate_preds(
            model, tokenizer, samples,
            store_cls,
            sel_map[sel_type],
            recomputer, blender, max_new_tokens,
        )
        entry["rougeL"] = rouge.compute(predictions=preds, references=refs)["rougeL"]
        t3_dirty = True
        print(f"  {key} rougeL = {entry['rougeL']:.4f}")

    if t3_dirty:
        with open(t3_path, "w") as f:
            json.dump(t3, f, indent=2)
        print(f"\nSaved updated {t3_path}")

    return t12, t3


# ── Run the repair ─────────────────────────────────────────────────────────
t12_results, t3_results = repair_rouge(
    RESULTS_DIR,
    mistral_model, mistral_tok, mn_samples,
    mistral_recomputer, mistral_blender,
    max_new_tokens=128,
)

print_tables(t12_results, t3_results)


In [ ]:
print_tables(t12_results, t3_results)

summary = {'table1_table2': t12_results, 'table3': t3_results}
summary_path = f'{RESULTS_DIR}/eval_summary.json'
with open(summary_path, 'w') as f:
    json.dump(summary, f, indent=2)

print(f'\nFull results saved to {RESULTS_DIR}/')
print(f'  table1_table2.json  -- per-ratio TTFT + ROUGE-L')
print(f'  table3.json         -- selector x store ROUGE-L grid')
print(f'  eval_summary.json   -- combined summary')
